In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.5 MB/s eta 0:00:00


In [2]:
import torch
print(torch.cuda.is_available())

True


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
dataset_path = "/content/drive/MyDrive/pub_checker/dataset"

## Create YAML File

In [5]:
import yaml

data = {
    "path": "/content/drive/MyDrive/pub_checker/dataset",
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",

    "names": {
        0: "NYC_Correct",
        1: "NYC_Incorrect",
        2: "BP_Correct",
        3: "BP_Incorrect",
        4: "SK_Correct",
        5: "SK_Incorrect",
        6: "YORP_Correct",
        7: "YORP_Incorrect",
    }

}
with open("/content/drive/MyDrive/pub_checker/data.yaml", "w") as f:
    yaml.dump(data, f)

print("data.yaml file created.")

data.yaml file created.


## Train YOLO Model

In [6]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [7]:
# pre-trained YOLO model
model = YOLO("yolov8s.pt")

In [ ]:
# first training
model.train(
    data="/content/drive/MyDrive/pub_checker/data.yaml",
    project="/content/drive/MyDrive/pub_checker/runs",
    name="logo_model_v1",
    epochs=100,
    imgsz=640,
    batch=8,
    patience=15,
    device=0,
    workers=2,
    cache=True,
    amp=True
)

Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/pub_checker/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=logo_model_v2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patien

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b38157660c0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [ ]:
best_model = YOLO("/content/drive/MyDrive/pub_checker/runs/logo_model_v2/weights/best.pt")

In [ ]:
# longer training (epoch 200, patience 30)
model.train(
    data="/content/drive/MyDrive/pub_checker/data.yaml",
    project="/content/drive/MyDrive/pub_checker/runs",
    name="logo_model_v2",
    epochs=200,
    imgsz=800,
    batch=8,
    patience=30,
    device=0,
    workers=2,
    cache=True,
    amp=True
)

## Metrics

In [ ]:
print("Metrics for Pre-trained YOLOv8 model")
metrics = best_model.val(
    data="/content/drive/MyDrive/pub_checker/data.yaml",
    plots=True
)


Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,127,132 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.6±0.4 ms, read: 50.9±21.3 MB/s, size: 198.0 KB)
val: Scanning /content/drive/MyDrive/pub_checker/dataset/labels/val.cache... 110 images, 15 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 125/125 32.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.5it/s 3.2s
                   all        125        220      0.847      0.977      0.909      0.876
           NYC_Correct         51         51      0.842          1      0.945      0.945
         NYC_Incorrect         59         59      0.893      0.993      0.902      0.846
            BP_Correct         62         62      0.809          1      0.929      0.929
          BP_Incorrect         48         48      0.846      0.914      0.859      0.784
Speed: 2.9ms preprocess, 6

In [ ]:
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

RESULTS:
mAP50: 0.9086
mAP50-95: 0.8760
Precision: 0.8475
Recall: 0.9767
